# Equity Research Report Generator
**Spring 2026 — LLM Term Project**

Run cells top to bottom. Cell (5) generates the report; cell (7) exports it to PDF.

In [ ]:
# (1) Install dependencies
!pip -q install langgraph langchain-core langchain-community openai pydantic tenacity rich faiss-cpu sentence-transformers yfinance pypdf pandas python-dotenv markdown requests

In [ ]:
# (2) Project paths — change PROJECT_ROOT if you move the folder
import os, sys

PROJECT_ROOT = r"C:\Users\manoj\OneDrive\Manu\Domain knowlege\7.Financial Engineering\1.Financial engineering\19.Spring 2026\2. Large Language model\project"
NOTEBOOK_DIR = os.path.join(PROJECT_ROOT, "notebook")
DATA_DIR     = os.path.join(PROJECT_ROOT, "data")

# Add notebook folder to Python path so imports resolve
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

# Create dirs if they don't exist yet
os.makedirs(NOTEBOOK_DIR, exist_ok=True)
os.makedirs(DATA_DIR,     exist_ok=True)

print(f"Notebook : {NOTEBOOK_DIR}")
print(f"Data     : {DATA_DIR}")

In [ ]:
# (3) Load secrets from .env
# Copy .env.example → .env and fill in your keys. Never commit .env to git.
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=os.path.join(PROJECT_ROOT, ".env"), override=True)

# Confirm keys loaded (values masked)
for key in ["OPENROUTER_API_KEY", "WP_SITE_URL", "WP_USERNAME", "WP_APP_PASSWORD"]:
    val = os.getenv(key, "")
    print(f"{key}: {'✓ set' if val else '✗ MISSING'}")

In [ ]:
# (4) Import the graph
from orchestrator import run_report
from rich import print as rprint
rprint('[bold green]Graph ready ✓[/bold green]')

In [ ]:
# (5) Run — Goldman Sachs
#
# Expected data layout:
#   data\gs_filings\FRY9C_2380443_20250930.csv
#   data\gs_filings\FFIEC102_2380443_20250930.csv
#   data\gs_filings\2024-annual-report.pdf

TICKER       = 'GS'
FILING_DATE  = '20250930'
CSV_FOLDER   = os.path.join(DATA_DIR, 'gs_filings')
PDF_PATHS    = [os.path.join(DATA_DIR, 'gs_filings', '2024-annual-report.pdf')]

report = run_report(
    ticker      = TICKER,
    pdf_paths   = PDF_PATHS,
    csv_folder  = CSV_FOLDER,
    filing_date = FILING_DATE,
)

print(report[:3000])   # preview first 3000 chars

In [ ]:
# (6) Save markdown to data\
md_file = os.path.join(DATA_DIR, f"{TICKER}_equity_report.md")

with open(md_file, 'w', encoding='utf-8') as f:
    f.write(report)

rprint(f"[bold]Markdown saved → {md_file}[/bold]")

In [ ]:
# (7) Export to PDF via pandoc + xelatex (MiKTeX)
import subprocess

pdf_file = os.path.join(DATA_DIR, f"{TICKER}_equity_report.pdf")

result = subprocess.run([
    "pandoc", md_file,
    "-o",  pdf_file,
    "--pdf-engine=xelatex",
    "-V",  "geometry:margin=1in",
    "-V",  "fontsize=11pt",
    "-V",  "mainfont=Times New Roman",
    "-V",  "colorlinks=true",
    "-V",  "urlcolor=blue",
    "--highlight-style=tango",
], capture_output=True, text=True)

if result.returncode == 0:
    rprint(f"[bold green]PDF saved → {pdf_file}[/bold green]")
else:
    rprint(f"[red]pandoc error:[/red]\n{result.stderr}")

In [ ]:
# (8) Publish to WordPress.com
#
# Publishes the report as a DRAFT by default — review in WP Admin before going live.
# Change status="publish" to go live immediately.
#
# Pre-requisite: fill in WP_SITE_URL, WP_USERNAME, WP_APP_PASSWORD in your .env
# Get app password: WP Admin → Users → Profile → Application Passwords → Add New

from publisher import publish_report

POST_TITLE = f"{TICKER} Equity Research — {'Buy' if 'buy' in report.lower() else 'Hold'} | Bulge Bracket"

result = publish_report(
    report_md = report,
    title     = POST_TITLE,
    status    = "draft",       # change to "publish" when ready
    tags      = [TICKER, "Equity Research", "Financial Analysis"],
    excerpt   = report.split("\n\n")[1][:200] if "\n\n" in report else "",
)

rprint(f"[bold green]✓ Draft created[/bold green]")
rprint(f"  Post ID  : {result['post_id']}")
rprint(f"  View URL : {result['url']}")
rprint(f"  Edit URL : {result['edit_url']}")